In [ ]:
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from ai_image_models.data import CIFAR10Dataset
from ai_image_models.models import FlowMLP
from ai_image_models.learner import Learner, train_classifier
from torchvision.utils import make_grid

data_dir = Path("../01_train/data/cifar10")
ds = CIFAR10Dataset(data_dir)
clf = train_classifier(ds, img_shape=(32, 32, 3), n_classes=10)


flow = FlowMLP(img_shape=(32, 32, 3))
flow.load_state_dict(
    torch.load(
        "../01_train/flow_mlp.pt",
        map_location="cpu",
        weights_only=True,
    )
)
learner = Learner(flow)

clf

In [ ]:
clf.features(ds.x[0:1].to("mps")).shape

In [ ]:
from pyexpat import features


real_features = clf.features(ds.x[:2048])
real_features.shape

In [ ]:
# gen_images = learner.generate(n=2048, trajectory=False)
# print(gen_images.shape)

In [ ]:
# gen_features = clf.features(gen_images)
# gen_features.shape

In [ ]:
def gen_feats(n_steps):
    gen_images = learner.generate(
        n=2048,
        steps=n_steps,
        trajectory=False
    )
    gen_features = clf.features(gen_images)
    return gen_features

In [ ]:
def fid(f1, f2):
    m1, m2 = f1.mean(0), f2.mean(0)
    c1, c2 = torch.cov(f1.T), torch.cov(f2.T)
    eig = torch.linalg.eigvals(c1 @ c2).real.clamp(min=0)

    return (
        ((m1 - m2) ** 2).sum()
        + torch.trace(c1 + c2)
        - 2 * eig.sqrt().sum()
    ).item()

In [ ]:
fid(real_features, gen_feats(10))

In [ ]:
from tqdm import tqdm

X = [1, 2, 4, 16, 32, 64, 100, 200]
Y = []

for n_steps in tqdm(X):
    FID = fid(real_features, gen_feats(n_steps))
    Y.append(FID)
print(Y[-1])

In [ ]:
import matplotlib.pyplot as plt

plt.plot(X, Y)

In [ ]:

imgs = learner.generate(n=64,steps=10)

grid = make_grid(imgs.permute(0, 3, 1, 2), nrow=8)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis("off")
plt.show()